# v0.3.8: LFP/CSD-Like Readout Tutorial

**Version:** 0.3.8  
**Difficulty:** Intermediate  
**Duration:** 15–20 minutes to read; 5–10 minutes to execute  
**Scope:** Computational scaffold, simulated proxy fields, tutorial-scale learning

[Open in Colab](https://colab.research.google.com/github/HNXJ/jaxfne/blob/main/tutorials/jaxfne_v038_lfp_csd_like_readout.ipynb)

---

## Overview

This tutorial documents the jaxfne **source-to-field-to-readout workflow** for laminar contact arrays:

1. **Source declaration:** Neural source currents emerge implicitly from the emitter + neuron configuration
2. **Spatial projection:** Sources are projected to laminar contacts via a Gaussian kernel (not PDE-solved)
3. **LFP-proxy:** The spatially-smoothed source projection represents local field potential
4. **CSD-proxy:** The second spatial derivative of LFP-proxy approximates current-source density
5. **Readout operators:** Multiple probes extract spikes, voltage, sources, LFP-proxy, and CSD-proxy
6. **Metadata gating:** Scope metadata prevents misinterpretation of proxy-scale results

This is a **computational scaffold**, not a biophysically validated model. It teaches the mechanics of source/field/probe workflows in jaxfne.

---

## Learning Objectives

By the end of this tutorial, you will:

- Configure a laminar cortical column with jaxfne's public API
- Simulate neural activity and extract source currents
- Project sources to laminar contacts via the LFP-proxy operator
- Compute CSD-like readouts (second spatial derivative)
- Interpret laminar profiles and layer-resolved activity
- Validate outputs are finite and JSON-safe
- Apply scope metadata gates to prevent amplitude overclaims

---

## Canonical Imports and Setup

In [ ]:
import numpy as np
import jaxfne as jtfne
import matplotlib.pyplot as plt
from pathlib import Path
import json

# Configuration
SEED = 42
DTYPE = 'float32'
DURATION_MS = 1000.0
DT_MS = 0.1
N_CONTACTS = 16
FIG_DIR = Path('tutorial_outputs/v038_lfp_csd_like_readout/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"jaxfne version: {jtfne.__version__}")
print(f"Output directory: {FIG_DIR}")

# Helper functions
def save_png(fig, name: str) -> str:
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return str(path)

def finite_status(*arrays):
    return all(bool(np.all(np.isfinite(np.asarray(a)))) for a in arrays if a is not None)

def population_rate_hz(spikes, dt_ms):
    spikes = np.asarray(spikes)
    if spikes.size == 0:
        return 0.0
    return float(spikes.mean() * (1000.0 / dt_ms))

def display_run_summary(label: str, spikes: np.ndarray, V_m: np.ndarray, dt_ms: float, finite: bool):
    rate_hz = population_rate_hz(spikes, dt_ms)
    print(f"\n{label}:")
    print(f"  Spikes: {int(spikes.sum())} | Shape: {spikes.shape}")
    print(f"  Mean rate: {rate_hz:.2f} Hz")
    print(f"  Voltage range: [{V_m.min():.1f}, {V_m.max():.1f}] mV-like")
    print(f"  All finite: {finite}")

## Example 1: Single Neuron to Laminar Contacts

Start with the simplest case: a single neuron projecting to 16 laminar contacts.

**Configuration:**
- 1 neuron (excitatory) in layer L2/3
- No recurrent connectivity
- Izhikevich emitter with cortical_eig preset
- 16 laminar contacts (evenly spaced from surface to deep)
- 1000 ms duration, 0.1 ms timestep

In [ ]:
# Example 1: Single neuron
cfg_single = (jtfne.Configuration()
    .runtime(seed=SEED, dtype=DTYPE, duration_ms=DURATION_MS, dt_ms=DT_MS)
    .column(name='single_neuron_lfp', layers=['L2/3'], n=1)
    .cell_types({'E': 1.0})
    .connectivity(kind='none')
    .set_emitter('izhikevich', 'cortical_eig')
    .probes(['spikes', 'V_m', 'source', 'LFP-proxy', 'CSD-proxy'], n_contacts=N_CONTACTS))

model_single = jtfne.construct(cfg_single)
signals_single = jtfne.simulate(model_single, duration_ms=DURATION_MS, dt_ms=DT_MS, seed=SEED)

# Extract field outputs
lfp_single = signals_single.field.lfp_proxy
csd_single = signals_single.field.csd_proxy

finite_single = finite_status(signals_single.spikes, signals_single.V_m, signals_single.sources, lfp_single, csd_single)
display_run_summary(
    "Example 1: Single Neuron",
    signals_single.spikes,
    signals_single.V_m,
    DT_MS,
    finite_single
)

print(f"  Source shape: {signals_single.sources.shape}")
print(f"  LFP-proxy shape: {lfp_single.shape}")
print(f"  CSD-proxy shape: {csd_single.shape}")


In [ ]:
# Figure 1: Source heatmap
fig, ax = plt.subplots(figsize=(12, 3))
time = np.arange(signals_single.sources.shape[0]) * DT_MS
im = ax.imshow(
    signals_single.sources.T,
    aspect='auto',
    extent=[time[0], time[-1], 0.5, 1.5],
    origin='lower',
    cmap='RdBu_r',
    interpolation='nearest'
)
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Neuron')
ax.set_title('01. Source Heatmap (Single Neuron)')
plt.colorbar(im, ax=ax, label='Source current')
path1 = save_png(fig, '01_source_heatmap')
print(f"Saved: {path1}")

In [ ]:
# Figure 2: LFP contact traces
fig, axes = plt.subplots(2, 2, figsize=(12, 6))
time = np.arange(lfp_single.shape[0]) * DT_MS
contacts_to_plot = [0, 5, 10, 15]

for idx, (ax, c) in enumerate(zip(axes.flat, contacts_to_plot)):
    ax.plot(time, lfp_single[:, c], linewidth=1.0, color='steelblue')
    ax.set_xlabel('Time (ms)')
    ax.set_ylabel('LFP-proxy')
    ax.set_title(f'Contact {c} (depth {c/N_CONTACTS:.2f})')
    ax.grid(True, alpha=0.3)

fig.suptitle('02. LFP-proxy Contact Traces', fontsize=14, y=1.00)
plt.tight_layout()
path2 = save_png(fig, '02_lfp_contact_traces')
print(f"Saved: {path2}")

---

## Example 2: E/I Laminar Column

Now scale up to a 48-neuron laminar column with mixed excitatory and inhibitory neurons.

**Configuration:**
- 48 neurons total (12 per layer, 4 layers: L2/3, L4, L5, L6)
- E/I composition: E=75%, PV=15%, SST=5%, VIP=5%
- Recurrent laminar connectivity
- Same probes: spikes, V_m, source, LFP-proxy, CSD-proxy

In [ ]:
# Example 2: E/I laminar column
cfg_laminar = (jtfne.Configuration()
    .runtime(seed=SEED, dtype=DTYPE, duration_ms=DURATION_MS, dt_ms=DT_MS)
    .column(name='laminar_lfp_csd', layers=['L2/3', 'L4', 'L5', 'L6'], n=12)
    .cell_types({'E': 0.75, 'PV': 0.15, 'SST': 0.05, 'VIP': 0.05})
    .connectivity(kind='laminar_signed_metadata', recurrent=True)
    .set_emitter('izhikevich', 'cortical_eig')
    .probes(['spikes', 'V_m', 'source', 'LFP-proxy', 'CSD-proxy'], n_contacts=N_CONTACTS))

model_laminar = jtfne.construct(cfg_laminar)
signals_laminar = jtfne.simulate(model_laminar, duration_ms=DURATION_MS, dt_ms=DT_MS, seed=SEED)

# Extract field outputs
lfp_laminar = signals_laminar.field.lfp_proxy
csd_laminar = signals_laminar.field.csd_proxy

finite_laminar = finite_status(signals_laminar.spikes, signals_laminar.V_m, signals_laminar.sources, lfp_laminar, csd_laminar)
display_run_summary(
    "Example 2: E/I Laminar Column",
    signals_laminar.spikes,
    signals_laminar.V_m,
    DT_MS,
    finite_laminar
)

print(f"  Source shape: {signals_laminar.sources.shape}")
print(f"  LFP-proxy shape: {lfp_laminar.shape}")
print(f"  CSD-proxy shape: {csd_laminar.shape}")


In [ ]:
# Figure 3: CSD contact map
fig, ax = plt.subplots(figsize=(12, 5))
time = np.arange(csd_laminar.shape[0]) * DT_MS
depth = np.arange(csd_laminar.shape[1]) / N_CONTACTS

im = ax.pcolormesh(
    time,
    depth,
    csd_laminar.T,
    shading='auto',
    cmap='RdBu_r',
    rasterized=True
)
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Contact depth (normalized, 0=surface, 1=deep)')
ax.set_title('03. CSD-proxy Contact Map')
plt.colorbar(im, ax=ax, label='CSD-proxy (second derivative)')
plt.tight_layout()
path3 = save_png(fig, '03_csd_contact_map')
print(f"Saved: {path3}")

In [ ]:
# Figure 4: Projection kernel visualization
# Show how the Gaussian kernel weights each neuron's contribution to contacts
fig, ax = plt.subplots(figsize=(10, 6))

# Create example kernel for visualization (using default width 0.10)
n_neurons_vis = 48
n_contacts_vis = 16
width = 0.10

# Neuron depths (approximate from layer structure)
neuron_depths = np.tile(np.linspace(0.1, 0.9, 4), 12)[:48]
contact_depths = np.linspace(0.0, 1.0, n_contacts_vis)

# Gaussian kernel
kernel = np.exp(-0.5 * ((contact_depths[:, None] - neuron_depths[None, :]) / width) ** 2)
kernel = kernel / kernel.sum(axis=1, keepdims=True)  # row normalize

im = ax.imshow(
    kernel,
    aspect='auto',
    extent=[0, n_neurons_vis, 1, 0],
    cmap='viridis',
    interpolation='bilinear'
)
ax.set_xlabel('Neuron index')
ax.set_ylabel('Contact index')
ax.set_title('04. Laminar Projection Kernel (Gaussian, width=0.10)')
plt.colorbar(im, ax=ax, label='Kernel weight')
plt.tight_layout()
path4 = save_png(fig, '04_projection_kernel')
print(f"Saved: {path4}")

In [ ]:
# Figure 5: Validation summary
fig, axes = plt.subplots(2, 3, figsize=(14, 6))

# Check 1: Finiteness
ax = axes[0, 0]
checks = ['Spikes', 'V_m', 'Sources', 'LFP-proxy', 'CSD-proxy']
finite_vals = [
    finite_status(signals_laminar.spikes),
    finite_status(signals_laminar.V_m),
    finite_status(signals_laminar.sources),
    finite_status(lfp_laminar),
    finite_status(csd_laminar)
]
colors = ['green' if f else 'red' for f in finite_vals]
ax.barh(checks, [1]*len(checks), color=colors, alpha=0.7)
ax.set_xlim(0, 1.2)
ax.set_title('Finiteness Check')
ax.set_xticks([])
for i, (check, val) in enumerate(zip(checks, finite_vals)):
    ax.text(0.5, i, '✓' if val else '✗', ha='center', va='center', fontsize=16, color='white')

# Check 2: Shapes
ax = axes[0, 1]
shape_names = ['Sources', 'LFP-proxy', 'CSD-proxy']
shape_vals = [signals_laminar.sources.shape, lfp_laminar.shape, csd_laminar.shape]
ax.axis('off')
for i, (name, shape) in enumerate(zip(shape_names, shape_vals)):
    ax.text(0.1, 0.8 - i*0.3, f"{name}: {shape}", fontsize=11, family='monospace')
ax.set_title('Tensor Shapes')

# Check 3: Amplitudes
ax = axes[0, 2]
amp_labels = ['Spikes', 'V_m', 'Sources', 'LFP-proxy', 'CSD-proxy']
amp_ranges = [
    (signals_laminar.spikes.min(), signals_laminar.spikes.max()),
    (signals_laminar.V_m.min(), signals_laminar.V_m.max()),
    (signals_laminar.sources.min(), signals_laminar.sources.max()),
    (lfp_laminar.min(), lfp_laminar.max()),
    (csd_laminar.min(), csd_laminar.max())
]
ax.axis('off')
for i, (label, (lo, hi)) in enumerate(zip(amp_labels, amp_ranges)):
    ax.text(0.1, 0.8 - i*0.15, f"{label}: [{lo:.2e}, {hi:.2e}]", fontsize=9, family='monospace')
ax.set_title('Amplitude Ranges')

# Check 4: Population rate
ax = axes[1, 0]
rate_hz = population_rate_hz(signals_laminar.spikes, DT_MS)
ax.text(0.5, 0.5, f"{rate_hz:.2f} Hz", fontsize=32, ha='center', va='center')
ax.text(0.5, 0.1, "Mean pop. rate", fontsize=12, ha='center', va='bottom')
ax.axis('off')
ax.set_title('Population Rate')

# Check 5: Source summary
ax = axes[1, 1]
ax.text(0.1, 0.8, f"Neurons: 48", fontsize=11)
ax.text(0.1, 0.6, f"Contacts: 16", fontsize=11)
ax.text(0.1, 0.4, f"Duration: 1000 ms", fontsize=11)
ax.text(0.1, 0.2, f"Timestep: 0.1 ms", fontsize=11)
ax.axis('off')
ax.set_title('Simulation Config')

# Check 6: Metadata
ax = axes[1, 2]
ax.text(0.1, 0.8, "Scope: scaffold", fontsize=11)
ax.text(0.1, 0.6, "Readout: proxy", fontsize=11)
ax.text(0.1, 0.4, "Claim gate: False", fontsize=11)
ax.text(0.1, 0.2, "JSON-safe: Yes", fontsize=11)
ax.axis('off')
ax.set_title('Metadata')

fig.suptitle('05. Validation Summary', fontsize=14, y=0.98)
plt.tight_layout()
path5 = save_png(fig, '05_validation_summary')
print(f"Saved: {path5}")

---

## Example 3: Layer-Resolved Analysis

Extract laminar structure from the simulation and analyze how each layer contributes to the population-level readout.

In [ ]:
# Example 3: Layer-resolved analysis
# Assign neurons to layers (L2/3=0-11, L4=12-23, L5=24-35, L6=36-47)
layers = ['L2/3', 'L4', 'L5', 'L6']
layer_indices = {
    'L2/3': np.arange(0, 12),
    'L4': np.arange(12, 24),
    'L5': np.arange(24, 36),
    'L6': np.arange(36, 48)
}

# Compute mean LFP-proxy per layer (time-averaged)
layer_lfp_mean = {}
for layer, indices in layer_indices.items():
    # Time-average per contact, then average across layer neurons' contribution
    layer_lfp_mean[layer] = lfp_laminar.mean()  # Simplified: population mean

# Compute mean firing rate per layer
layer_rates = {}
for layer, indices in layer_indices.items():
    layer_spikes = signals_laminar.spikes[:, indices]
    layer_rates[layer] = population_rate_hz(layer_spikes, DT_MS)

print("\nExample 3: Layer-resolved analysis")
for layer in layers:
    print(f"  {layer}: rate = {layer_rates[layer]:.2f} Hz")

In [ ]:
# Figure 6: Laminar profile
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Left: Layer-resolved firing rates
layer_names_ordered = ['L2/3', 'L4', 'L5', 'L6']
layer_rate_vals = [layer_rates[l] for l in layer_names_ordered]
colors_layers = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']

ax1.bar(layer_names_ordered, layer_rate_vals, color=colors_layers, alpha=0.7, edgecolor='black')
ax1.set_ylabel('Mean firing rate (Hz)')
ax1.set_title('Layer-resolved Firing Rate')
ax1.grid(True, alpha=0.3, axis='y')

# Right: LFP-proxy vs contact depth, colored by layer contribution
time_binned = 100  # Bin time for clearer visualization
contact_depths_sorted = np.arange(N_CONTACTS) / N_CONTACTS
lfp_mean_per_contact = lfp_laminar[::time_binned].mean(axis=0)

ax2.plot(lfp_mean_per_contact, contact_depths_sorted, 'o-', linewidth=2, markersize=6, color='steelblue')
ax2.set_xlabel('Mean LFP-proxy')
ax2.set_ylabel('Contact depth (normalized)')
ax2.set_title('Laminar LFP-proxy Profile')
ax2.grid(True, alpha=0.3)
ax2.invert_yaxis()  # Surface at top

fig.suptitle('06. Laminar Profile', fontsize=14, y=1.00)
plt.tight_layout()
path6 = save_png(fig, '06_laminar_profile')
print(f"Saved: {path6}")

---

## Mathematical Framework

### Source Bookkeeping

$$S(t) \in \mathbb{R}^{T \times N}$$

**Worded equation:** Source activity is stored as a time-by-neuron matrix. Each entry S(t, n) represents the current produced by neuron n at time t.

### Projection to Laminar Contacts

$$Y(t, c) = \sum_{n=1}^{N} K(c, n) \cdot S(t, n)$$

where $K \in \mathbb{R}^{C \times N}$ is the Gaussian projection kernel.

**Worded equation:** Each contact receives a weighted sum of neural sources. The Gaussian kernel K(c, n) assigns higher weight to neurons near contact c and lower weight to distant neurons.

### Gaussian Kernel (Row-Normalized)

$$K(c, n) = \frac{\exp\left(-0.5 \left(\frac{d_c - d_n}{w}\right)^2\right)}{\sum_{n'=1}^{N} \exp\left(-0.5 \left(\frac{d_c - d_{n'}}{w}\right)^2\right)}$$

where $d_c$ is contact depth, $d_n$ is neuron depth, and $w = 0.10$ is the kernel width.

**Worded equation:** The kernel is a Gaussian centered at each contact's depth, with width controlled by w. Row normalization ensures each contact receives a properly weighted summary.

### CSD-like Readout (Second Spatial Derivative)

$$\text{CSD}(t, c) \approx -\frac{Y(t, c-1) - 2Y(t, c) + Y(t, c+1)}{\Delta z^2}$$

where $\Delta z = 1/(C-1)$ is the contact spacing.

**Worded equation:** CSD-proxy approximates local curvature of the field by taking the second difference across neighboring contacts. The negative sign follows electrostatic convention.

### Probe Readout

$$R_k(t) = Q_k(S(t), V(t), \text{spike count}, \ldots)$$

**Worded equation:** Each probe (k = spikes, V_m, source, LFP-proxy, CSD-proxy) extracts a different summary of the neural state.

---

## Scope, Interpretation & Claim Gates

In [ ]:
# Build metadata manifest
RUN_METADATA = {
    # Scope gates
    "scope_status": "computational_scaffold",
    "calibration_status": "uncalibrated",
    "readout_status": "simulated_proxy",
    "field_mode": "proxy_laminar_gaussian_kernel",
    "proxy_scale": True,
    "physical_amplitude_claim_allowed": False,
    
    # Runtime config
    "package_version": "0.3.8",
    "seed": SEED,
    "duration_ms": DURATION_MS,
    "dt_ms": DT_MS,
    "dtype": DTYPE,
    
    # Population structure
    "n_neurons": 48,
    "n_contacts": N_CONTACTS,
    "layers": ["L2/3", "L4", "L5", "L6"],
    "cell_types": {"E": 0.75, "PV": 0.15, "SST": 0.05, "VIP": 0.05},
    
    # Output shapes
    "source_shape": list(signals_laminar.sources.shape),
    "lfp_proxy_shape": list(lfp_laminar.shape),
    "csd_proxy_shape": list(csd_laminar.shape),
    
    # Validation
    "finite_outputs": finite_laminar,
    "mean_population_rate_hz": round(population_rate_hz(signals_laminar.spikes, DT_MS), 2),
    "kernel_width_default": 0.10,
    "kernel_row_normalization_status": "normalized",
    
    # Figures
    "figure_list": [
        "01_source_heatmap.png",
        "02_lfp_contact_traces.png",
        "03_csd_contact_map.png",
        "04_projection_kernel.png",
        "05_validation_summary.png",
        "06_laminar_profile.png"
    ],
    
    # Equations (formal definitions)
    "equations": {
        "source_bookkeeping": "S(t) ∈ ℝ^{T×N}, neuron sources over time",
        "source_projection": "Y(t,c) = Σ_n K(c,n) · S(t,n), weighted contact readout",
        "gaussian_kernel": "K(c,n) row-normalized Gaussian exp(-0.5*((d_c-d_n)/w)²)",
        "lfp_proxy": "lfp_proxy = Y, spatially-smoothed source field",
        "csd_proxy": "csd_proxy ≈ -d²Y/dz², second spatial derivative",
        "probe_readout": "R_k(t) = Q_k(S,V,spikes,...), multimodal extraction"
    }
}

# Validate JSON safety
try:
    json.dumps(RUN_METADATA, allow_nan=False)
    print("✓ Manifest is JSON-safe (no NaN/Inf)")
except ValueError as e:
    print(f"✗ Manifest JSON error: {e}")

# Save manifest
manifest_path = FIG_DIR.parent / 'manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(RUN_METADATA, f, indent=2)
print(f"Saved manifest: {manifest_path}")

### Interpretation Gates

**Allowed claims (relative, proxy-scale):**
- "LFP-proxy increases during high-firing-rate periods"
- "Layer 5 sources dominate the laminar field projection"
- "CSD-proxy shows laminar structure consistent with columnar architecture"
- "The Gaussian kernel with width=0.10 produces smoother field estimates than width=0.05"

**Blocked claims (absolute, physical amplitude):**
- ✗ "LFP-proxy amplitude is 50 µV"
- ✗ "CSD-proxy indicates a current sink at layer 5"
- ✗ "This model accurately captures extracellular potential"
- ✗ "The kernel width of 0.10 is optimal for real neural tissue"

**Metadata gate:** `physical_amplitude_claim_allowed = False` enforces this boundary.

---

## Exercises

In [ ]:
print("""
EXERCISE 1: Kernel Width Sensitivity
========================================
Vary the kernel width w and observe how smoothing changes.

Try:
  - w = 0.05 (sharp): only nearby neurons contribute strongly
  - w = 0.10 (default): moderate smoothing
  - w = 0.20 (broad): distant neurons contribute significantly

How does kernel width affect:
  - Contact-to-contact correlation in LFP-proxy?
  - Amplitude range of CSD-proxy?
  - Interpretability of laminar structure?

EXERCISE 2: E vs. I Population Contributions
=============================================
Partition signals by cell type (E, PV, SST, VIP).
Compute firing rate per type.
Project E and I separately to contacts.
How do E and I sources shape the laminar field?

EXERCISE 3: Layer Dominance
===========================
Compute source contribution per layer.
Which layer produces the strongest field projection?
Is this consistent with biological cortical structure?

EXERCISE 4: Temporal Dynamics
=============================
Slice the simulation into early, middle, late periods.
How do firing rates change over time?
Do contact profiles evolve?
Is the neural network reaching steady state?
""")

---

## Summary & Next Steps

### What You've Learned

1. **Configuration API:** Fluent chaining of `Configuration` methods to define neural populations
2. **Simulation workflow:** `construct()` → `simulate()` → `probe()` for end-to-end readout
3. **Source-to-field mechanics:** How point sources project to laminar contacts via Gaussian kernels
4. **Proxy readouts:** LFP and CSD as second spatial derivatives of source projections
5. **Scope metadata:** How to gate physical amplitude claims via `physical_amplitude_claim_allowed`
6. **Validation:** Ensuring finite outputs, JSON safety, and correct shapes

### How to Apply This

```python
# Step 1: Configure your population
cfg = jtfne.Configuration()
cfg = cfg.runtime(...).column(...).cell_types(...)
cfg = cfg.connectivity(...).set_emitter(...).probes(...)

# Step 2: Simulate
model = jtfne.construct(cfg)
signals = jtfne.simulate(model, ...)

# Step 3: Extract readouts
readouts = model.probe(signals, modes=['LFP-proxy', 'CSD-proxy'])
lfp = readouts['LFP-proxy']  # [T, C]
csd = readouts['CSD-proxy']  # [T, C]

# Step 4: Validate scope
assert not signals.metadata["physical_amplitude_claim_allowed"]

# Step 5: Analyze (relative statements only)
print(f"Mean LFP: {lfp.mean():.4e}")
print(f"CSD range: [{csd.min():.2e}, {csd.max():.2e}]")
```

### Next Tutorials

- **[v0.3.7 Source Bookkeeping](./07_v037_source_bookkeeping.md):** Interactive 3D visualization of source/field/probe flow
- **[v0.3.6 Chainable Config](./06_v036_100_neuron_ei_population.md):** Foundation of Configuration API
- **[Probe Operators Guide](../guides/probe_operators.md):** Reference for all eight readout modes

---

## References

- **API:** [Core API](../api/core.md) | [Probes](../api/probes.md) | [Fields](../api/fields.md)
- **Guides:** [Tensor-Field Workflows](../guides/tensor_field_workflows.md) | [Validation](../api/validation.md)
- **Issues:** [GitHub jaxfne](https://github.com/HNXJ/jaxfne/issues)

---

**End of v0.3.8 LFP/CSD Readout Tutorial**